# Inputs: prompts, tools, and data

JAZ only has a single LLM primitive — the `invoke` agent loop. In basic usage, its signature
is just `invoke(**inputs)` with arbitrary keyword arguments `**inputs`. An LLM
writes code in a Python REPL loop until it returns as though it is dynamically
filling in the body of a function that received `inputs`.

While all existing frameworks distinguish between different kinds of "agent inputs" (e.g., prompt, tools, and data),
JAZ makes no such distinction because
[no such distinction is needed and the distinction can even be harmful](https://arxiv.org/abs/2609.26891).
Every argument you pass to `invoke` — the prompt, a tool, or really any Python object in your program —
is just an *input* in JAZ's taxonomy.
**JAZ `invoke` treats all inputs the same way: they all become variables in the REPL and are all rendered into the LLM's context by JAZ's renderer**.

## Setup

To set up this notebook, install JAZ, provide an API key, and configure a model:

In [ ]:
%pip install -q jaz-lang pandas

In [1]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [2]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## All agent inputs are treated equally

JAZ's `invoke` has the simplest possible interface: it simply takes **arbitrary keyword arguments**,
`invoke(name1=value1, name2=value2, ...)`.
An input can be any Python object and you can give it whatever name you want.

Every keyword argument `name=value` you pass to `invoke`:

1. becomes a **variable named `name` in the agent's REPL**;
2. is **shown in the agent's prompt**, so the model sees it.

> *Unlike other frameworks, JAZ `invoke` has no special `user_prompt`/`task` or `tools` arguments. JAZ makes no distinction between different kinds of inputs because such distinction only adds unnecessary complexity such as tool schemas and tool registration boilerplate.*

To understand this, we need a helper hook that lets us inspect the prompt and agent trajectory.
*You do not need to understand it now — hooks are introduced in [tutorial 3](03-hooks-events-and-effects.ipynb)*.

In [ ]:
from jaz import invoke
from jaz.hooks import Hook

class Logger(Hook):
    """Prints the prompt and agent trajectory."""

    def on_llm_query_enter(self, event):
        label = "USER PROMPT" if event.iteration == 0 else "REPL OUTPUT"
        print(f"--- {label} BEGINS ---\n{event.messages[-1]['content']}\n--- {label} ENDS ---")

    def on_llm_query_complete(self, event):
        print(f"--- LLM RESPONSE BEGINS ---\n{event.response.content}\n--- LLM RESPONSE ENDS ---")

In the following example, we give `invoke` two inputs: a string and a pandas DataFrame
containing information about 344 individual penguins from the
[Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) dataset.
Here, the string describes a simple data analysis task in English.

In [ ]:
import pandas as pd

# Palmer Penguins: 344 rows; each row is one penguin individual
penguins = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")

with Logger():
    answer = invoke(
        task="Compute the average body mass of each species.",
        penguins=penguins,
    )

answer

--- USER PROMPT BEGINS ---
<task type="str">
Compute the average body mass of each species.
</task>
<penguins type="DataFrame">
    species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen            39.1           18.7              181.0   
1    Adelie  Torgersen            39.5           17.4              186.0   
2    Adelie  Torgersen            40.3           18.0              195.0   
3    Adelie  Torgersen             NaN            NaN                NaN   
4    Adelie  Torgersen            36.7           19.3              193.0   
..      ...        ...             ...            ...                ...   
339  Gentoo     Biscoe             NaN            NaN                NaN   
340  Gentoo     Biscoe            46.8           14.3              215.0   
341  Gentoo     Biscoe            50.4           15.7              222.0   
342  Gentoo     Biscoe            45.2           14.8              212.0   
343  Gentoo     Biscoe            49

--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Compute average body_mass_g grouped by species and return the result.

result = (
    penguins.groupby("species", as_index=False)["body_mass_g"]
    .mean()
    .sort_values("species")
)

result = result.set_index("species")["body_mass_g"]
return result
--- LLM RESPONSE ENDS ---


species
Adelie       3700.662252
Chinstrap    3733.088235
Gentoo       5076.016260
Name: body_mass_g, dtype: float64

Because the framework treats all inputs identically,
`task` and `penguins` are formatted the *same way* in the user prompt. As you can see in the output above,
each is formatted as an XML element named after itself, with its type as an attribute and content as its body.
Nothing in the framework marks one as the instruction and the other as the data
— the agent simply received two named values and figured out what to do. It saw a section
in its prompt titled `<task>` and interpreted its contents as the task, and it saw a section
named `<penguins>` and interpreted it as the data the task was referring to.

Here the `penguins` DataFrame's string representation in the prompt is just its `str()`, which omits
most of its rows. However, the object itself is fully available to the agent as the
`penguins` variable in its REPL.
So the LLM can just write regular Python code on the `penguins` DataFrame to conduct the requested data analysis.

Note that `task` is not the name of an argument in `invoke`'s signature — `invoke`'s signature is quite literally
`invoke(**inputs)` with arbitrary keyword arguments. Instead, we just happened to name the input containing
the instructions "`task`". You can
use any other name that the LLM will understand (e.g., `question` or `instructions`), since it just names
the section in the LLM's prompt that displays that input. The following example renames `task`
to `question` and `penguins` to `data` and agent behavior is unaffected.

In [5]:
# Giving different names yields the same result - it just renames the section titles
# in the prompt and the variables in the REPL.
with Logger():
    answer = invoke(
        question="What is the mean body mass of each species?",
        data=penguins,
    )

answer

--- USER PROMPT BEGINS ---
<question type="str">
What is the mean body mass of each species?
</question>
<data type="DataFrame">
    species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen            39.1           18.7              181.0   
1    Adelie  Torgersen            39.5           17.4              186.0   
2    Adelie  Torgersen            40.3           18.0              195.0   
3    Adelie  Torgersen             NaN            NaN                NaN   
4    Adelie  Torgersen            36.7           19.3              193.0   
..      ...        ...             ...            ...                ...   
339  Gentoo     Biscoe             NaN            NaN                NaN   
340  Gentoo     Biscoe            46.8           14.3              215.0   
341  Gentoo     Biscoe            50.4           15.7              222.0   
342  Gentoo     Biscoe            45.2           14.8              212.0   
343  Gentoo     Biscoe            4

--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Compute mean body mass (body_mass_g) grouped by species, ignoring NaNs.

df = data
result = df.groupby("species", dropna=False)["body_mass_g"].mean().sort_index()
return result
--- LLM RESPONSE ENDS ---


species
Adelie       3700.662252
Chinstrap    3733.088235
Gentoo       5076.016260
Name: body_mass_g, dtype: float64

## A tool is an input that happens to be callable

Since passing a function as an input to `invoke` exposes it as a variable in the agent's REPL,
it becomes a *tool* the agent can call in the style of programmatic tool calling.
JAZ renders it in the user prompt as its signature and docstring, equivalent
to the tool spec in traditional tool-calling agents.

In the following example, we give `invoke` a tool `same_distribution_p_value` that performs a
particular statistical analysis (Mann-Whitney U test).
Notice how `prompt` doesn't need to mention the function because JAZ's input renderer already renders it
in the user prompt.

In [ ]:
import math

def same_distribution_p_value(a: list[float], b: list[float]) -> float:
    """Mann-Whitney U test: the two-sided p-value for the null hypothesis that
    two samples come from the same distribution. Missing values (NaN) are ignored."""
    a, b = [x for x in a if x == x], [y for y in b if y == y]
    n1, n2 = len(a), len(b)
    u = sum((x > y) + (x == y) / 2 for x in a for y in b)
    z = (abs(u - n1 * n2 / 2) - 0.5) / math.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)
    return math.erfc(z / math.sqrt(2))


with Logger():
    reply = invoke(
        prompt="Is there statistically significant evidence that Adelie and "
               "Chinstrap penguins differ in body mass? Give me the p-value.",
        penguins=penguins,
        same_distribution_p_value=same_distribution_p_value,
    )

reply

--- USER PROMPT BEGINS ---
<prompt type="str">
Is there statistically significant evidence that Adelie and Chinstrap penguins differ in body mass?
</prompt>
<penguins type="DataFrame">
    species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen            39.1           18.7              181.0   
1    Adelie  Torgersen            39.5           17.4              186.0   
2    Adelie  Torgersen            40.3           18.0              195.0   
3    Adelie  Torgersen             NaN            NaN                NaN   
4    Adelie  Torgersen            36.7           19.3              193.0   
..      ...        ...             ...            ...                ...   
339  Gentoo     Biscoe             NaN            NaN                NaN   
340  Gentoo     Biscoe            46.8           14.3              215.0   
341  Gentoo     Biscoe            50.4           15.7              222.0   
342  Gentoo     Biscoe            45.2           14.8  

--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Filter to Adelie vs Chinstrap, extract body_mass_g (dropping NaNs),
# run a two-sided Mann-Whitney U test via same_distribution_p_value, return p-value.

adelie_masses = penguins.loc[penguins["species"] == "Adelie", "body_mass_g"].dropna().astype(float).tolist()
chinstrap_masses = penguins.loc[penguins["species"] == "Chinstrap", "body_mass_g"].dropna().astype(float).tolist()

p_value = same_distribution_p_value(adelie_masses, chinstrap_masses)

return p_value
--- LLM RESPONSE ENDS ---


0.4856732660841894

From the output above, you can see that inputs that are functions are treated the same way
as any other input: rendered into the user prompt and made available as a variable in the REPL.
The only difference here is that JAZ's renderer no longer renders its `str()`, but instead
its signature and docstring. Thus, writing a tool in JAZ does not involve writing a schema
or registering an explicit tool object — just pass your function directly into `invoke`.

## JAZ's input renderer: How an input is displayed to the model

You can think of JAZ's input renderer as Python's `str()` but modified to suit the LLM behind `invoke`.
So far we've seen that JAZ's input renderer directly uses Python's `str()` for strings and objects like
DataFrame instances, but for functions it uses their signature and docstring. The general rules for rendering
*any* Python object are summarized in the following table — we've omitted some details for simplicity.

| The input is… | The model is shown |
| :------------ | :----------------- |
| a function or method | its signature and docstring |
| a class | its constructor's signature and the class's docstring, then its public members |
| a module | its docstring, then its public members |
| an object that does not define `__str__` or `__repr__` | its class docstring, then its public members |
| anything else *(default)* | its `str()`, which calls `__str__` if defined otherwise `__repr__` |

> *Note: If the rendering is longer than 50,000 characters, then its middle part is cut out and replaced with `[...N characters omitted...]`. This conditional truncation is applied uniformly to all inputs.*

Examples we have seen so far:
- `task` and `prompt` are strings. Since Python's `str` type defines `__str__`, they fall under the
  "anything else *(default)*" category, so JAZ simply calls Python's `str()` on them.
- `penguins` is a `pd.DataFrame` object. Since the `pd.DataFrame` class defines `__str__`,
  it's also in the "anything else *(default)*" category and JAZ calls Python's `str()` on it.
- `same_distribution_p_value` is a function, aka it falls under the "function or method" category,
  and JAZ renders it as its signature and docstring.
  
As an example of an "object that 'does not print itself'", we can define a custom class that doesn't
define `__str__` or `__repr__`, and pass an instance into `invoke`:

In [7]:
class Colony:
    """The penguins observed on one island."""

    def __init__(self, island: str, penguins: pd.DataFrame):
        self.island = island
        self.penguins = penguins

    def share(self, species: str) -> float:
        """Percentage of this colony's penguins that belong to `species`."""
        return 100 * float((self.penguins["species"] == species).mean())


with Logger():
    reply = invoke(
        prompt="What share of the penguins on Dream are Chinstrap, in percent?",
        dream=Colony("Dream", penguins[penguins["island"] == "Dream"]),
    )

reply

--- USER PROMPT BEGINS ---
<prompt type="str">
What share of the penguins on Dream are Chinstrap, in percent?
</prompt>
<dream type="Colony">
`dream` [Colony]: The penguins observed on one island.
  - `island` [str]
  - `penguins` [DataFrame]
  - `share(species: str) -> float`: Percentage of this colony's penguins that belong to `species`.
</dream>
The input variables `prompt`, `dream` are available in your REPL with descriptions given above.
They are not automatically available to `invoke()` calls you make, so making them available to sub-invokes requires passing them explicitly.

--- USER PROMPT ENDS ---


--- LLM RESPONSE BEGINS ---
# <brief plan for next step>
# Compute dream.share("Chinstrap") and return as the requested percent
result = dream.share("Chinstrap")
return result
--- LLM RESPONSE ENDS ---


54.83870967741935

Since the custom class `Colony` doesn't define `__str__` or `__repr__`,
JAZ instead showed the `dream` colony's public attributes
(`island` and `penguins`) and public method `share()`. This told the LLM how to use the `dream` object
so that the LLM knew it had to call `dream.share("Chinstrap")` to complete the task. 

> _Note: To provide a custom rendering for your custom object, simply define `__str__` or `__repr__` on your custom class. But be careful as that **replaces** JAZ's default rendering describing the public interface of the class, and the LLM may not know the interface of your custom class so it may no longer know how to interact with your object. In that case, make sure your custom `__str__` or `__repr__` also includes the class's interface. (This wasn't an issue with `pd.DataFrame` because LLMs have seen plenty of `pandas` in its training data.)_

## Summary

- **One input mechanism.** Every input — be it a prompt, a tool, or a custom object — is a keyword
  argument `name=value` to `invoke`. It becomes a variable named `name` in the REPL and is shown in the prompt
  under a section titled `name`.
- **JAZ's input renderer.** Think of it as Python's `str()`, but for the LLM behind `invoke` to read.
  - **A function's display is its signature and docstring.** Thus, to give the agent a tool, just give `invoke` the function
    implementing the tool. There's no boilerplate like creating a tool object, generating a tool spec, or
    registering the tool on the agent.
  - **An object that doesn't define its own display through `__str__` or `__repr__` is rendered as its class docstring and its public members**, so that the agent knows how to use it.
  - **An object that does define its `__str__` or `__repr__` is rendered as its Python `str()`**,
    so you can define `__str__` or `__repr__` on your custom class to provide a custom rendering.
    If you want to change the rendering of an existing object, you can wrap it in such a class.

---

**Next:** [Recursive subagents and scope](02-subagents-and-scope.ipynb) — an agent can call `invoke` itself, which raises the question of what a *subagent* can see in its context and have access to in its REPL.